#  Classification d'images Caltech-256 avec CNN et Vision Transformers
## Version optimisée avec train/val split stratifié et fine-tuning progressif

##  Installation et imports

In [1]:
# Installation des packages nécessaires
!pip install torch torchvision datasets transformers pillow matplotlib scikit-learn -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, models
from datasets import load_dataset
from transformers import ViTForImageClassification, ViTImageProcessor
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import time
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

# Configuration du device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  Device utilisé: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Mémoire disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

  Device utilisé: cuda
   GPU: Tesla T4
   Mémoire disponible: 15.64 GB


##  Chargement et préparation du dataset Caltech-256

**IMPORTANT:** Caltech-256 contient **257 classes** (256 catégories + 1 classe "clutter")

Nous allons créer un split train/val stratifié à partir du dataset train original.

In [ ]:
# Charger le dataset Caltech-256 depuis Hugging Face
print(" Chargement du dataset Caltech-256...")
dataset = load_dataset("ilee0022/Caltech-256", trust_remote_code=True)

# Afficher les informations sur le dataset
print(f"\n Informations sur le dataset:")
print(f"   Train: {len(dataset['train'])} images")
if 'validation' in dataset:
    print(f"   Validation: {len(dataset['validation'])} images")
if 'test' in dataset:
    print(f"   Test: {len(dataset['test'])} images (non utilisé)")

# Vérifier le nombre de classes uniques
train_labels = [item['label'] for item in dataset['train']]
num_classes = max(train_labels) + 1
print(f"   Nombre de classes détecté: {num_classes}")
print(f"   Label min: {min(train_labels)}, Label max: {max(train_labels)}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ilee0022/Caltech-256' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


 Chargement du dataset Caltech-256...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/452M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

data/validation-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24791 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3061 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2755 [00:00<?, ? examples/s]


 Informations sur le dataset:
   Train: 24791 images
   Test: 3061 images
   Nombre de classes détecté: 258
   Label min: 1, Label max: 257


In [ ]:
# Configuration du dataset
USE_FULL_DATASET = True  # Utiliser tout le dataset pour de meilleurs résultats
VAL_SPLIT = 0.2  # 20% des données pour la validation

print(f"\n Configuration du dataset:")
print(f"   Utilisation du dataset complet: {USE_FULL_DATASET}")
print(f"   Split validation: {VAL_SPLIT * 100}%")

# Créer un split stratifié train/val
from collections import Counter

# Récupérer les indices et labels du train
all_indices = list(range(len(dataset['train'])))
all_labels = [item['label'] for item in dataset['train']]

# Vérifier la distribution des classes
label_counts = Counter(all_labels)
print(f"\n Nombre d'exemples par classe (min-max): {min(label_counts.values())}-{max(label_counts.values())}")

# Split stratifié pour maintenir la distribution des classes
train_indices, val_indices = train_test_split(
    all_indices, 
    test_size=VAL_SPLIT, 
    stratify=all_labels,
    random_state=42
)

print(f"\n Split stratifié créé:")
print(f"   Train: {len(train_indices)} images")
print(f"   Validation: {len(val_indices)} images")

# Créer les sous-ensembles
train_dataset = dataset['train'].select(train_indices)
val_dataset = dataset['train'].select(val_indices)

# Vérifier la distribution stratifiée
train_labels_split = [item['label'] for item in train_dataset]
val_labels_split = [item['label'] for item in val_dataset]
print(f"\n Classes uniques dans train: {len(set(train_labels_split))}")
print(f" Classes uniques dans val: {len(set(val_labels_split))}")

 Mode rapide activé: utilisation de 5000 images par split
 Dataset final: 5000 train, 3061 test


##  Transformations avec Data Augmentation forte

In [ ]:
# Transformations pour les modèles CNN avec Data Augmentation forte
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),  # Plus agressif
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1))
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Dataset wrapper pour PyTorch
class HFImageDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image']
        label = item['label']
        
        # Convertir en RGB si nécessaire
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Créer les datasets PyTorch
train_pytorch_dataset = HFImageDataset(train_dataset, transform=train_transforms)
val_pytorch_dataset = HFImageDataset(val_dataset, transform=val_transforms)

# DataLoaders optimisés
BATCH_SIZE = 32
NUM_WORKERS = 4  # Augmenté pour un chargement plus rapide

train_loader = DataLoader(
    train_pytorch_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=NUM_WORKERS, 
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_pytorch_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS, 
    pin_memory=True
)

print(f" DataLoaders créés:")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Num workers: {NUM_WORKERS}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")

 DataLoaders créés: batch_size=32


##  Création des modèles

In [6]:
def create_model(model_name, num_classes, pretrained=True):
    """
    Crée un modèle CNN ou ViT pré-entraîné
    
    Args:
        model_name: 'resnet50', 'vgg16', 'efficientnet_b0', 'vit'
        num_classes: nombre de classes (257 pour Caltech-256)
        pretrained: utiliser les poids pré-entraînés
    """
    print(f"  Création du modèle {model_name.upper()} pour {num_classes} classes...")
    
    if model_name == 'resnet50':
        if pretrained:
            model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        else:
            model = models.resnet50(weights=None)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, num_classes)
        
    elif model_name == 'vgg16':
        if pretrained:
            model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        else:
            model = models.vgg16(weights=None)
        num_ftrs = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(num_ftrs, num_classes)
        
    elif model_name == 'efficientnet_b0':
        if pretrained:
            model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        else:
            model = models.efficientnet_b0(weights=None)
        num_ftrs = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(num_ftrs, num_classes)
        
    elif model_name == 'vit':
        model = ViTForImageClassification.from_pretrained(
            'google/vit-base-patch16-224-in21k',
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
    else:
        raise ValueError(f"Modèle {model_name} non supporté")
    
    return model

print(" Fonction de création de modèles prête")

 Fonction de création de modèles prête


##  Fonction d'entraînement avec Fine-tuning progressif

**Stratégie:**
1. Phase 1 (epochs 1-5): Backbone gelé, entraînement de la dernière couche uniquement
2. Phase 2 (epochs 6-10): Tout le modèle dégelé avec fine-tuning

In [ ]:
def train_model(model_name, num_epochs=10, learning_rate=0.001, use_progressive_finetuning=True):
    """
    Entraîne un modèle avec fine-tuning progressif
    
    Args:
        model_name: 'resnet50', 'vgg16', 'efficientnet_b0', 'vit'
        num_epochs: nombre total d'epochs
        learning_rate: taux d'apprentissage initial
        use_progressive_finetuning: utiliser le fine-tuning progressif (freeze puis unfreeze)
    """
    print(f"\n{'='*80}")
    print(f"Entraînement de {model_name.upper()}")
    print(f"{'='*80}\n")
    
    # Créer le modèle avec le bon nombre de classes
    model = create_model(model_name, num_classes=num_classes, pretrained=True)
    
    # Déplacer le modèle vers le GPU
    try:
        model = model.to(device)
        _ = model(torch.randn(1, 3, 224, 224).to(device))
        print(f" Modèle chargé sur {device}")
    except Exception as e:
        print(f" Erreur lors du chargement sur {device}: {e}")
        torch.cuda.empty_cache()
        model = model.to(device)
    
    # Configuration du fine-tuning progressif
    freeze_epochs = num_epochs // 2 if use_progressive_finetuning else 0
    
    if use_progressive_finetuning:
        print(f"\n Fine-tuning progressif activé:")
        print(f"   Phase 1 (epochs 1-{freeze_epochs}): Backbone gelé")
        print(f"   Phase 2 (epochs {freeze_epochs+1}-{num_epochs}): Fine-tuning complet")
        
        # Geler le backbone initialement
        for name, param in model.named_parameters():
            if model_name == 'vit':
                if 'classifier' not in name:
                    param.requires_grad = False
            else:
                if 'fc' not in name and 'classifier' not in name:
                    param.requires_grad = False
    
    # Définir la fonction de perte et l'optimiseur avec weight decay
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    # Optimiseur Adam avec weight_decay pour la régularisation
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), 
        lr=learning_rate,
        weight_decay=1e-4,
        betas=(0.9, 0.999)
    )
    
    # Scheduler CosineAnnealingLR pour un decay smooth du learning rate
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, 
        T_max=num_epochs, 
        eta_min=1e-6
    )
    
    # Alternative: ReduceLROnPlateau plus agressif
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    #     optimizer, 
    #     mode='min', 
    #     factor=0.3, 
    #     patience=1, 
    #     min_lr=1e-6
    # )
    
    # Historique d'entraînement
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'lr': []
    }
    
    start_time = time.time()
    best_acc = 0.0
    
    # Boucle d'entraînement
    for epoch in range(num_epochs):
        print(f"\n Epoch {epoch+1}/{num_epochs}")
        current_lr = optimizer.param_groups[0]['lr']
        print(f"   Learning Rate: {current_lr:.6f}")
        
        # Dégeler le backbone après freeze_epochs
        if use_progressive_finetuning and epoch == freeze_epochs:
            print(f"\n  PHASE 2: Dégelage du backbone pour fine-tuning complet")
            for param in model.parameters():
                param.requires_grad = True
            
            # Réinitialiser l'optimiseur avec un learning rate plus petit
            optimizer = optim.AdamW(
                model.parameters(), 
                lr=learning_rate * 0.1,
                weight_decay=1e-4
            )
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, 
                T_max=num_epochs - freeze_epochs, 
                eta_min=1e-6
            )
        
        # Phase d'entraînement
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        train_pbar = tqdm(train_loader, desc='Training', leave=False)
        for images, labels in train_pbar:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            if model_name == 'vit':
                outputs = model(images).logits
            else:
                outputs = model(images)
            
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Gradient clipping pour éviter l'explosion des gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            train_pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})
        
        train_loss = running_loss / len(train_loader)
        train_acc = 100. * correct / total
        
        # Phase de validation
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc='Validation', leave=False)
            for images, labels in val_pbar:
                images, labels = images.to(device), labels.to(device)
                
                if model_name == 'vit':
                    outputs = model(images).logits
                else:
                    outputs = model(images)
                
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
                
                val_pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})
        
        val_loss /= len(val_loader)
        val_acc = 100. * correct / total
        
        # Mettre à jour l'historique
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)
        
        # Afficher les résultats
        print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"   Val Loss: {val_loss:.4f}   | Val Acc: {val_acc:.2f}%")
        
        # Sauvegarder le meilleur modèle
        if val_acc > best_acc:
            best_acc = val_acc
            print(f"    Nouveau meilleur modèle! Val Accuracy: {best_acc:.2f}%")
        
        # Scheduler step
        scheduler.step()
        # Pour ReduceLROnPlateau, utiliser: scheduler.step(val_loss)
    
    training_time = time.time() - start_time
    print(f"\n Entraînement terminé en {training_time/60:.2f} minutes")
    print(f" Meilleure accuracy de validation: {best_acc:.2f}%")
    
    return model, history, training_time

##  Fonction d'évaluation et de visualisation

In [ ]:
def evaluate_model(model, model_name, val_loader):
    """
    Évalue le modèle sur le set de validation et calcule les métriques détaillées
    """
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc='Evaluation'):
            images = images.to(device)
            
            if model_name == 'vit':
                outputs = model(images).logits
            else:
                outputs = model(images)
            
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    # Calculer les métriques
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='weighted', zero_division=0
    )
    
    print(f"\n Résultats pour {model_name.upper()}:")
    print(f"   Accuracy: {accuracy*100:.2f}%")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall: {recall:.4f}")
    print(f"   F1-Score: {f1:.4f}")
    
    return accuracy, precision, recall, f1

def plot_training_history(history, model_name):
    """
    Affiche les courbes de loss, accuracy et learning rate
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss
    axes[0].plot(epochs, history['train_loss'], label='Train Loss', marker='o', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], label='Val Loss', marker='s', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title(f'{model_name.upper()} - Loss', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(epochs, history['train_acc'], label='Train Acc', marker='o', linewidth=2)
    axes[1].plot(epochs, history['val_acc'], label='Val Acc', marker='s', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy (%)', fontsize=12)
    axes[1].set_title(f'{model_name.upper()} - Accuracy', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
    
    # Learning Rate
    axes[2].plot(epochs, history['lr'], marker='o', color='green', linewidth=2)
    axes[2].set_xlabel('Epoch', fontsize=12)
    axes[2].set_ylabel('Learning Rate', fontsize=12)
    axes[2].set_title(f'{model_name.upper()} - Learning Rate', fontsize=14, fontweight='bold')
    axes[2].set_yscale('log')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
print(" Fonctions d'évaluation et de visualisation prêtes")

##  Entraînement des modèles

In [ ]:
# Configuration de l'entraînement optimisée
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
USE_PROGRESSIVE_FINETUNING = True  # Fine-tuning progressif (freeze puis unfreeze)

print(f" Configuration de l'entraînement:")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Learning Rate: {LEARNING_RATE}")
print(f"   Progressive Fine-tuning: {USE_PROGRESSIVE_FINETUNING}")
print(f"   Weight Decay: 1e-4")
print(f"   Scheduler: CosineAnnealingLR")
print(f"   Label Smoothing: 0.1")

# Résultats globaux
results = {}

### ResNet50

In [ ]:
# Entraîner ResNet50
model_resnet, history_resnet, time_resnet = train_model(
    model_name='resnet50',
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    use_progressive_finetuning=USE_PROGRESSIVE_FINETUNING
)

# Visualiser
plot_training_history(history_resnet, 'ResNet50')

# Évaluer sur le set de validation
acc_resnet, prec_resnet, rec_resnet, f1_resnet = evaluate_model(model_resnet, 'resnet50', val_loader)
results['ResNet50'] = {
    'accuracy': acc_resnet,
    'precision': prec_resnet,
    'recall': rec_resnet,
    'f1': f1_resnet,
    'time': time_resnet
}

### VGG16

In [ ]:
# Entraîner VGG16
model_vgg, history_vgg, time_vgg = train_model(
    model_name='vgg16',
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    use_progressive_finetuning=USE_PROGRESSIVE_FINETUNING
)

# Visualiser
plot_training_history(history_vgg, 'VGG16')

# Évaluer sur le set de validation
acc_vgg, prec_vgg, rec_vgg, f1_vgg = evaluate_model(model_vgg, 'vgg16', val_loader)
results['VGG16'] = {
    'accuracy': acc_vgg,
    'precision': prec_vgg,
    'recall': rec_vgg,
    'f1': f1_vgg,
    'time': time_vgg
}

### EfficientNet-B0

In [ ]:
# Entraîner EfficientNet-B0
model_eff, history_eff, time_eff = train_model(
    model_name='efficientnet_b0',
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    use_progressive_finetuning=USE_PROGRESSIVE_FINETUNING
)

# Visualiser
plot_training_history(history_eff, 'EfficientNet-B0')

# Évaluer sur le set de validation
acc_eff, prec_eff, rec_eff, f1_eff = evaluate_model(model_eff, 'efficientnet_b0', val_loader)
results['EfficientNet-B0'] = {
    'accuracy': acc_eff,
    'precision': prec_eff,
    'recall': rec_eff,
    'f1': f1_eff,
    'time': time_eff
}

### Vision Transformer (ViT)

In [ ]:
# Entraîner ViT
model_vit, history_vit, time_vit = train_model(
    model_name='vit',
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    use_progressive_finetuning=USE_PROGRESSIVE_FINETUNING
)

# Visualiser
plot_training_history(history_vit, 'ViT')

# Évaluer sur le set de validation
acc_vit, prec_vit, rec_vit, f1_vit = evaluate_model(model_vit, 'vit', val_loader)
results['ViT'] = {
    'accuracy': acc_vit,
    'precision': prec_vit,
    'recall': rec_vit,
    'f1': f1_vit,
    'time': time_vit
}

##  Comparaison finale des modèles

In [ ]:
import pandas as pd

# Créer un DataFrame des résultats
df_results = pd.DataFrame(results).T
df_results['time_min'] = df_results['time'] / 60
df_results = df_results[['accuracy', 'precision', 'recall', 'f1', 'time_min']]
df_results.columns = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Time (min)']

print("\n" + "="*80)
print(" RÉSULTATS FINAUX - Comparaison des modèles")
print("="*80)
print(df_results.to_string())
print("="*80)

# Visualisation comparative
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    df_results[metric].plot(kind='bar', ax=ax, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
    ax.set_title(f'{metric} par modèle', fontsize=12, fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_xlabel('Modèle')
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])
    
    # Ajouter les valeurs sur les barres
    for i, v in enumerate(df_results[metric]):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Trouver le meilleur modèle
best_model = df_results['Accuracy'].idxmax()
print(f"\n Meilleur modèle: {best_model} avec {df_results.loc[best_model, 'Accuracy']:.4f} d'accuracy")

##  Sauvegarde des modèles

In [ ]:
# Sauvegarder les modèles
import os

os.makedirs('models', exist_ok=True)

torch.save(model_resnet.state_dict(), 'models/resnet50_caltech256.pth')
torch.save(model_vgg.state_dict(), 'models/vgg16_caltech256.pth')
torch.save(model_eff.state_dict(), 'models/efficientnet_b0_caltech256.pth')
model_vit.save_pretrained('models/vit_caltech256')

print(" Modèles sauvegardés dans le dossier 'models/'")

##  Test de prédiction sur une image aléatoire

In [ ]:
def predict_image(model, model_name, image_tensor, true_label):
    """
    Prédit la classe d'une image
    """
    model.eval()
    with torch.no_grad():
        image_tensor = image_tensor.unsqueeze(0).to(device)
        
        if model_name == 'vit':
            outputs = model(image_tensor).logits
        else:
            outputs = model(image_tensor)
        
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        confidence, predicted = torch.max(probabilities, 1)
        
    return predicted.item(), confidence.item()

# Tester sur une image aléatoire du set de validation
import random

idx = random.randint(0, len(val_pytorch_dataset) - 1)
image, label = val_pytorch_dataset[idx]

# Afficher l'image
img_display = image.permute(1, 2, 0).cpu().numpy()
img_display = (img_display * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
img_display = np.clip(img_display, 0, 1)

plt.figure(figsize=(12, 3))
plt.subplot(1, 5, 1)
plt.imshow(img_display)
plt.title(f'Image validation\nVraie classe: {label}')
plt.axis('off')

# Prédictions de chaque modèle
models_dict = {
    'ResNet50': (model_resnet, 'resnet50'),
    'VGG16': (model_vgg, 'vgg16'),
    'EfficientNet': (model_eff, 'efficientnet_b0'),
    'ViT': (model_vit, 'vit')
}

for i, (name, (model, model_name)) in enumerate(models_dict.items(), 2):
    pred, conf = predict_image(model, model_name, image, label)
    correct = '✓' if pred == label else '✗'
    color = 'green' if pred == label else 'red'
    
    plt.subplot(1, 5, i)
    plt.text(0.5, 0.7, f'{name}', ha='center', fontsize=12, fontweight='bold')
    plt.text(0.5, 0.5, f'{correct}', ha='center', fontsize=30, color=color)
    plt.text(0.5, 0.3, f'Pred: {pred}\nConf: {conf*100:.1f}%', ha='center', fontsize=10)
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.axis('off')

plt.tight_layout()
plt.show()

##  Conclusion

Ce notebook compare 4 architectures de deep learning sur Caltech-256 avec une approche optimisée:

### Modèles comparés:
- **ResNet50**: Architecture résiduelle classique
- **VGG16**: Architecture convolutionnelle profonde
- **EfficientNet-B0**: Architecture optimisée pour l'efficacité
- **Vision Transformer (ViT)**: Architecture basée sur les transformers

### Améliorations implémentées:
1.  **Split stratifié Train/Val (80/20)**: Maintien de la distribution des classes
2.  **Fine-tuning progressif**: 
    - Phase 1: Backbone gelé (entraînement de la dernière couche)
    - Phase 2: Dégelage complet pour fine-tuning
3.  **Data Augmentation forte**:
    - RandomResizedCrop (scale 0.6-1.0)
    - Flips horizontal/vertical
    - ColorJitter agressif
    - Random rotation, affine, grayscale, erasing
4.  **Optimisation AdamW**: weight_decay=1e-4 pour régularisation
5.  **CosineAnnealingLR**: Decay smooth du learning rate
6.  **Label Smoothing**: Régularisation de 0.1
7.  **Gradient Clipping**: Évite l'explosion des gradients

### Métriques évaluées:
- Accuracy, Precision, Recall, F1-Score sur le set de validation
- Temps d'entraînement
- Convergence du learning rate

### Résultats:
Consultez la section "Comparaison finale des modèles" pour voir les performances détaillées de chaque architecture.